# Stage II (lesions) - 01. Data preparation for HSMoE-AUNet

One HSMoE-AUNet is trained per lesion type: microaneurysms (MA), haemorrhages (HE), hard
exudates (EX) and cotton-wool spots (CWS). This notebook builds the image-mask set of one
lesion type. Run it once per lesion (set `LESION`).

1. **Configuration** - lesion and folders.
2. **Pairing and filtering** - every fundus image is paired with its mask by file stem
   (lesion suffixes such as `_MA` or `_Hard_Exudates` are stripped from mask names). Pairs whose
   mask has no more positive pixels (grey value > 127, counted at the original resolution) than
   the lesion minimum are removed: MA 10, EX 30, HE 50, CWS 20 pixels. Empty masks give no
   positive gradient and push the model towards all-background output.
3. **Retinal crop** - image and mask are cropped to the bounding rectangle of the retinal disc
   (grey threshold 15, elliptical 11 x 11 closing and opening, 10-pixel margin); the mask is
   binarised and both are saved as PNG together with a manifest CSV.
4. **Split** - `train_test_split(test_size=0.30, random_state=42)` followed by
   `train_test_split(temp, test_size=0.33, random_state=42)` gives training, validation and test
   sets (test sets of 175 MA, 157 HE, 142 EX and 69 CWS images in the paper run).

## Step 1. Configuration

In [ ]:
from pathlib import Path

LESION = "MA"  # "MA" | "HE" | "EX" | "CWS"

# Expected raw layout (pooled FGADR, MAPLES-DR, IDRiD and e-ophtha image-mask pairs):
#   data/stage2_lesions/raw/<LESION>/images/<stem>.<ext>
#   data/stage2_lesions/raw/<LESION>/masks/<stem>[_<lesion suffix>].<ext>
DATA_ROOT = Path("data/stage2_lesions")
RAW_IMG_DIR = DATA_ROOT / "raw" / LESION / "images"
RAW_MASK_DIR = DATA_ROOT / "raw" / LESION / "masks"

# Outputs: cropped PNG pairs, manifest and split lists
OUT_DIR = DATA_ROOT / LESION
OUT_IMG_DIR = OUT_DIR / "images"
OUT_MASK_DIR = OUT_DIR / "masks"
MANIFEST = OUT_DIR / "manifest.csv"

In [ ]:
import pandas as pd

from hierarchiretina.stage2.hsmoe_aunet import get_lesion_config
from hierarchiretina.stage2.lesion_data import load_splits, prepare_lesion_dataset

cfg = get_lesion_config(LESION)
print(f"{cfg.name} ({cfg.full_name}): keep masks with > {cfg.min_lesion_pixels} positive pixels")
print("mask suffixes stripped for pairing:", cfg.mask_strip_suffixes)

## Step 2-3. Pairing, filtering, retinal crop and manifest

In [ ]:
manifest = prepare_lesion_dataset(LESION, RAW_IMG_DIR, RAW_MASK_DIR, OUT_IMG_DIR, OUT_MASK_DIR,
                                  MANIFEST)
manifest[["height", "width", cfg.pixel_col]].describe().round(0)

## Step 4. Split

The split is re-created from the manifest wherever it is needed (training, evaluation, overlap
check), so the CSV files written here are only a record. With the full data the sizes equal
those of the paper run (`check_sizes=True`).

In [ ]:
train_df, val_df, test_df = load_splits(MANIFEST, LESION, check_sizes=False)
print(f"train {len(train_df)} | val {len(val_df)} | test {len(test_df)} "
      f"(paper run: {cfg.expected_split_sizes})")

split_dir = OUT_DIR / "splits"
split_dir.mkdir(parents=True, exist_ok=True)
for name, df in (("train", train_df), ("val", val_df), ("test", test_df)):
    df[["stem"]].to_csv(split_dir / f"{name}.csv", index=False)